# AI-Powered POS Analytics - Exploration Notebook

**Personal learning project - All data is fictional**

This notebook explores the AI capabilities for Maple Corner Café's POS system.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generator import generate_pos_data
from src.models.forecaster import SalesForecaster
from src.models.basket_analyzer import BasketAnalyzer
from src.models.anomaly_detector import RefundAnomalyDetector
from src.models.llm_query import LLMQueryEngine

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

## 1. Generate Synthetic Data

Create 90 days of fictional transaction data for Maple Corner Café.

In [ ]:
# Generate data
print("Generating synthetic POS data...")
transactions, inventory = generate_pos_data(start_date="2024-01-01", num_days=90, random_seed=42)

print(f"\n📊 Generated Data:")
print(f"  Transaction line items: {len(transactions):,}")
print(f"  Unique transactions: {transactions['transaction_id'].nunique():,}")
print(f"  Date range: {transactions['date'].min()} to {transactions['date'].max()}")
print(f"  Products: {transactions['product_name'].nunique()}")
print(f"  Staff members: {transactions['staff_id'].nunique()}")
print(f"  Inventory items: {len(inventory)}")

In [ ]:
# Inspect sample transactions
transactions.head(10)

In [ ]:
# Basic statistics
print("📈 Sales Summary:")
print(f"  Total Sales: ${transactions[transactions['is_refund'] == False]['total'].sum():,.2f}")
print(f"  Total Refunds: ${abs(transactions[transactions['is_refund'] == True]['total'].sum()):,.2f}")
print(f"  Average Transaction: ${transactions.groupby('transaction_id')['total'].sum().mean():.2f}")
print(f"  Refund Rate: {transactions[transactions['is_refund'] == True]['transaction_id'].nunique() / transactions['transaction_id'].nunique():.2%}")

## 2. Sales Forecasting

Predict future sales based on historical patterns.

In [ ]:
# Train forecaster
forecaster = SalesForecaster()
forecaster.train(transactions)

print("✅ Forecaster trained!")

In [ ]:
# Peak hours
peak_hours = forecaster.get_peak_hours(top_n=10)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(peak_hours['hour'], peak_hours['avg_sales'], color='steelblue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Sales ($)')
ax.set_title('Average Sales by Hour', fontsize=14, fontweight='bold')
ax.set_xticks(peak_hours['hour'])
ax.set_xticklabels([f"{h:02d}:00" for h in peak_hours['hour']], rotation=45)
plt.tight_layout()
plt.show()

peak_hours

In [ ]:
# Busiest days
busiest_days = forecaster.get_busiest_days()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(busiest_days['day_name'], busiest_days['avg_sales'], color='coral')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Sales ($)')
ax.set_title('Average Sales by Day of Week', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

busiest_days

In [ ]:
# 7-day forecast
forecast = forecaster.forecast_daily(num_days=7)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(forecast['day_name'], forecast['forecast_sales'], color='green', alpha=0.7)
ax.set_xlabel('Day')
ax.set_ylabel('Forecast Sales ($)')
ax.set_title('7-Day Sales Forecast', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

forecast

## 3. Market Basket Analysis

Discover which products are frequently bought together.

In [ ]:
# Train basket analyzer
analyzer = BasketAnalyzer(min_support=0.05, min_confidence=0.5, min_lift=1.0)
analyzer.train(transactions)

stats = analyzer.get_statistics()
print("✅ Basket analyzer trained!")
print(f"\nStatistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Top association rules
top_rules = analyzer.get_top_rules(top_n=10)
print("🔗 Top 10 Association Rules:\n")
top_rules

In [ ]:
# Frequent pairs
pairs = analyzer.get_frequent_pairs(top_n=10)
print("👥 Most Frequent Product Pairs:\n")
pairs

In [ ]:
# Upsell recommendations
test_basket = ['Latte', 'Croissant']
recommendations = analyzer.get_upsell_recommendations(test_basket, top_n=5)

print(f"💡 Upsell Recommendations for {test_basket}:\n")
recommendations

## 4. Anomaly Detection

Identify unusual refund patterns.

In [ ]:
# Train anomaly detector
detector = RefundAnomalyDetector(contamination=0.1, random_state=42)
detector.train(transactions)

print("✅ Anomaly detector trained!")

In [ ]:
# Refund statistics
stats = detector.get_refund_statistics(transactions)

print("📊 Refund Statistics:\n")
for key, value in stats.items():
    if 'rate' in key:
        print(f"  {key}: {value:.2%}")
    elif 'amount' in key:
        print(f"  {key}: ${value:,.2f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Staff refund analysis
staff_analysis = detector.get_staff_refund_analysis(transactions)
print("👥 Staff Refund Analysis:\n")
staff_analysis

In [ ]:
# Anomalous refunds
anomalous = detector.get_anomalous_refunds(transactions)

if len(anomalous) > 0:
    print(f"🚨 Found {len(anomalous)} anomalous refund(s):\n")
    display(anomalous[['transaction_id', 'timestamp', 'staff_name', 'product_name', 'total', 'anomaly_explanation']])
else:
    print("✅ No anomalous refunds detected")

## 5. Natural Language Queries

Ask questions about the data in plain English.

In [ ]:
# Initialize query engine (mock mode)
engine = LLMQueryEngine(use_mock=True)

if engine.use_mock:
    print("🤖 Running in Mock Mode (no API key required)")
else:
    print("🌐 Connected to OpenAI API")

In [ ]:
# Example queries
questions = [
    "What were the top 5 products by revenue?",
    "Show me sales by hour of day",
    "Which staff member had the most transactions?",
    "How many refunds did we have?",
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print('='*80)
    
    response = engine.query(transactions, question)
    
    if response['success']:
        print(f"\nCode: {response['code']}")
        print(f"\nResult:\n{response['result_str']}")
    else:
        print(f"\nError: {response['error']}")

## 6. Additional Visualizations

In [ ]:
# Sales over time
daily_sales = transactions[transactions['is_refund'] == False].groupby('date')['total'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_sales['date'], daily_sales['total'], marker='o', linewidth=2, color='#1f77b4')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales ($)')
ax.set_title('Daily Sales Trend', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Category breakdown
category_sales = transactions[transactions['is_refund'] == False].groupby('category')['total'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
category_sales.plot(kind='bar', ax=ax, color='teal')
ax.set_xlabel('Category')
ax.set_ylabel('Total Revenue ($)')
ax.set_title('Revenue by Category', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 products
top_products = transactions[transactions['is_refund'] == False].groupby('product_name')['total'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
top_products.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Total Revenue ($)')
ax.set_ylabel('Product')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
- ✅ Synthetic data generation with realistic patterns
- ✅ Sales forecasting for staffing and inventory
- ✅ Market basket analysis for upselling
- ✅ Anomaly detection for fraud prevention
- ✅ Natural language query interface

See `CASE_STUDY.md` for detailed technical documentation.